In [ ]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "../../..")).expanduser().resolve()

# Convenience function for building repo-relative paths
def p(rel_path):
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

In [ ]:
# ============================================================
# Create temporary and output directories 
# ============================================================

DATA_DIR = p("data")
TMP_DIR = p("tmp")
OUT_DIR = p("data/output")

TMP_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

In [ ]:
import pandas as pd

# Each predictor's gene-specific calibration CSV carries all three
# Class_{REVEL,AM,MP2} columns, but the row sets differ per predictor
# (variants missing that predictor's score are dropped) -- so each needs
# its own file rather than reusing REVEL's with a different column name.
PREDICTOR_INPUTS = {
    "REVEL": ("controls_REVEL_GeneSpecific.csv", "Class_REVEL"),
    "AM": ("controls_AM_GeneSpecific.csv", "Class_AM"),
    "MP2": ("controls_MP2_GeneSpecific.csv", "Class_MP2"),
}

controls_by_predictor = {
    predictor: pd.read_csv(OUT_DIR / "predictor_calibration/gene_specific" / filename)
    for predictor, (filename, _) in PREDICTOR_INPUTS.items()
}

In [ ]:
import numpy as np

pathogenic = {"Pathogenic", "Likely pathogenic", "Likely Pathogenic","Pathogenic/Likely pathogenic"}
benign = {"Benign", "Likely benign", "Likely Benign","Benign/Likely benign"}

def collapse_path_ben(x):
    if pd.isna(x):
        return np.nan
    if x in pathogenic:
        return "Pathogenic"
    if x in benign:
        return "Benign"
    return np.nan


def gene_concordance(
    df,
    truth_col,
    pred_col,
    gene_col="Gene",
    uncertain_col = 'Class_REVEL'):
    tmp = df.copy()

    # Collapse truth / prediction
    tmp["truth_bin"] = tmp[truth_col].apply(collapse_path_ben)
    tmp["pred_bin"]  = tmp[pred_col].apply(collapse_path_ben)

    # Concordant
    tmp["concordant"] = (
        ((tmp["truth_bin"] == "Pathogenic") & (tmp["pred_bin"] == "Pathogenic")) |
        ((tmp["truth_bin"] == "Benign")     & (tmp["pred_bin"] == "Benign"))
    )

    # False negatives: Pathogenic → Benign
    tmp["FN"] = (
        (tmp["truth_bin"] == "Pathogenic") &
        (tmp["pred_bin"] == "Benign")
    )

    # False positives: Benign → Pathogenic
    tmp["FP"] = (
        (tmp["truth_bin"] == "Benign") &
        (tmp["pred_bin"] == "Pathogenic")
    )

    # Total discordance = FN + FP
    tmp["discordant"] = tmp["FN"] | tmp["FP"]

    # Uncertain
    tmp["uncertain"] = (
        tmp[uncertain_col] == "Uncertain"
    )

    return (
        tmp
        .groupby(gene_col)
        .agg(
            n_total=(gene_col, "size"),
            n_concordant=("concordant", "sum"),
            n_discordant=("discordant", "sum"),
            n_FN=("FN", "sum"),
            n_FP=("FP", "sum"),
            n_uncertain=("uncertain", "sum"),
        )
        .assign(
            concordance_pct=lambda x: 100 * x["n_concordant"] / x["n_total"],
            discordance_pct=lambda x: 100 * x["n_discordant"] / x["n_total"],
            FN_pct=lambda x: 100 * x["n_FN"] / x["n_total"],
            FP_pct=lambda x: 100 * x["n_FP"] / x["n_total"],
            uncertain_pct=lambda x: 100 * x["n_uncertain"] / x["n_total"],
        )
        .reset_index()
    )


In [ ]:
gene_conc = gene_concordance(
    df=controls_by_predictor["REVEL"],
    truth_col="clnsig_group_18_25",
    pred_col="Class_REVEL"
)


In [ ]:
import numpy as np
import pandas as pd

def compute_gene_stats(df, class_col):
    tmp = df.copy()
    tmp["truth_bin"] = tmp["clnsig_group_18_25"].apply(collapse_path_ben)
    tmp["pred_bin"] = tmp[class_col].apply(collapse_path_ben)

    # Strict discordance only: P↔B flips
    tmp["discordant"] = (
        ((tmp["truth_bin"] == "Pathogenic") & (tmp["pred_bin"] == "Benign")) |
        ((tmp["truth_bin"] == "Benign")     & (tmp["pred_bin"] == "Pathogenic"))
    )

    gene_stats = (
        tmp
        .groupby("Gene")
        .agg(
            n_total=("Gene", "size"),
            n_discordant=("discordant", "sum")
        )
        .reset_index()
    )

    total_discordant = gene_stats["n_discordant"].sum()
    total_variants = gene_stats["n_total"].sum()
    gene_stats["background_rate_loo"] = (
        (total_discordant - gene_stats["n_discordant"]) /
        (total_variants   - gene_stats["n_total"])
    )
    return gene_stats

gene_stats_by_predictor = {
    predictor: compute_gene_stats(controls_by_predictor[predictor], class_col)
    for predictor, (_, class_col) in PREDICTOR_INPUTS.items()
}

In [ ]:
from scipy.stats import binomtest

for gene_stats in gene_stats_by_predictor.values():
    gene_stats["p_value"] = gene_stats.apply(
        lambda r: binomtest(
            k=int(r["n_discordant"]),
            n=int(r["n_total"]),
            p=r["background_rate_loo"],
            alternative="greater"
        ).pvalue,
        axis=1
    )

In [ ]:
from statsmodels.stats.multitest import multipletests

for gene_stats in gene_stats_by_predictor.values():
    gene_stats["q_value"] = multipletests(
        gene_stats["p_value"],
        method="fdr_bh"
    )[1]


In [ ]:
for gene_stats in gene_stats_by_predictor.values():
    gene_stats["discordance_rate"] = (
        gene_stats["n_discordant"] / gene_stats["n_total"]
    )
    gene_stats["excess_discordance"] = (
        gene_stats["discordance_rate"] - gene_stats["background_rate_loo"]
    )


In [ ]:
OUTLIERS_by_predictor = {
    predictor: gene_stats[
        (gene_stats["n_total"] >= 10) &
        (gene_stats["q_value"] < 0.05) &
        (gene_stats["excess_discordance"] > 0)
    ].sort_values("excess_discordance", ascending=False)
    for predictor, gene_stats in gene_stats_by_predictor.items()
}

In [ ]:
from IPython.display import display

for predictor, outliers in OUTLIERS_by_predictor.items():
    print(f"--- {predictor} ---")
    display(outliers)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap

EMERALD_SEQ = LinearSegmentedColormap.from_list(
    "emerald_seq",
    [
        "#264D49",  
        "#43978D",  
        "#F9E07F",  
        "#F9AD6A",  
        "#D46C4E",
    ]
)

def plot_discordance(gene_stats, predictor_name):
    gene_stats = gene_stats.copy()
    gene_stats["neglog10_p"] = -np.log10(gene_stats["p_value"])

    # Define which genes get labels
    label_genes = gene_stats[
        (gene_stats["q_value"] < 0.05) &
        (gene_stats["n_total"] >= 10)
    ]

    fig, ax = plt.subplots(figsize=(7, 5))

    sc = ax.scatter(
        gene_stats["n_total"],
        gene_stats["discordance_rate"],
        c=gene_stats["neglog10_p"],
        cmap= EMERALD_SEQ,
        alpha=0.8
    )

    # Background / expected line
    ax.axhline(
        gene_stats["background_rate_loo"].mean(),
        color="red",
        linestyle="--",
        label="Expected discordance (LOO)"
    )

    # Label only significant genes
    for _, row in label_genes.iterrows():
        ax.annotate(
            row["Gene"],
            (row["n_total"], row["discordance_rate"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold"
        )

    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("-log10(p-value)")

    ax.set_xlabel("Variants per gene")
    ax.set_ylabel("Discordance rate")
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / "figures/extended_data_figure_5" / f"clinvar_discordance_per_gene_{predictor_name}.png",
        dpi=300, bbox_inches="tight"
    )
    plt.show()

for predictor, gene_stats in gene_stats_by_predictor.items():
    plot_discordance(gene_stats, predictor)